# 03 — Feature engineering and conditional promotion

**Goal:** Engineer time-windowed features that capture the *temporal context* of each transaction, train an enriched model, and promote it to production *only if* it beats the baseline on PR-AUC and F1.

**Why this notebook matters more than notebook 02 :**
- Real production fraud detection lives or dies on feature engineering, not model choice.
- This notebook exercises the *full* MLOps loop: train → log → compare → conditionally promote. Notebook 02 just trained and registered. This one decides whether production moves forward.

**The features we'll build:**
- **Velocity:** transaction counts in rolling 1h and 24h windows per sender
- **Aggregate amounts:** sum and historical mean of amounts per sender
- **Time-since:** hours since the same sender's previous transaction
- **Ratios:** amount-to-balance ratio and the "drains origin" boolean

**The promotion gate:**
The new model must beat the current `production` version on **both** PR-AUC and F1-at-calibrated-threshold. Otherwise it's logged for posterity but not promoted.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import logging
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import xgboost as xgb
from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from fraud_mlops.config import (
    DATA_INTERIM_DIR,
    FIGURES_DIR,
    LABEL_COLUMN,
    MODELS_DIR,
    RANDOM_SEED,
    TIME_COLUMN,
    ensure_dirs,
)
from fraud_mlops.data import load_paysim, split_features_label, time_based_split
from fraud_mlops.metrics import evaluate_at_threshold, find_threshold_for_precision
from fraud_mlops.tracking import setup_mlflow

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
ensure_dirs()
np.random.seed(RANDOM_SEED)

setup_mlflow(experiment_name="fraud_baseline")

## Load PaySim, sorted by time

Rolling windows depend on time order. The data must be sorted by `step` before any feature computation. PaySim *appears* sorted in the CSV, but never trust that — sort explicitly.

We also need `nameOrig` (sender ID) for grouping. The default `load_paysim()` drops it as an ID column, so we re-load without that filter.

In [ ]:
# Override the default `drop_leaky=True` because we need nameOrig for grouping.
# We'll still drop isFlaggedFraud manually.
df = load_paysim(drop_leaky=False).drop(columns=["isFlaggedFraud"])
df = df.sort_values(TIME_COLUMN).reset_index(drop=True)

print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print(f"\nTime range: step {df[TIME_COLUMN].min()} to {df[TIME_COLUMN].max()}")
print(f"Fraud rate: {df[LABEL_COLUMN].mean():.4%}")

## Engineer features

We compute features on the **full dataset** before splitting. Each row's features depend only on prior rows (its own past), so this is correct even though it spans the train/test boundary — a test row legitimately sees train rows that happened before it, because that's what would happen in production.

**The lookahead-leakage trap:** `closed="left"` on the rolling window ensures the current row is *not* counted in its own window. Without this, every feature would silently include "now" in "the last 24h" — a leak.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build temporal and ratio features per sender.

    Vectorized implementation: avoids per-group Python apply loops, which
    are catastrophically slow on PaySim's 6M+ unique senders.
    """
    df = df.copy()

    # Treat `step` (hours) as a datetime for time-based windows
    df["_ts"] = pd.to_datetime(df[TIME_COLUMN], unit="h", origin="2024-01-01")

    # Sort by sender, then time within sender. Crucial for the rolling logic below.
    df = df.sort_values(["nameOrig", "_ts"]).reset_index(drop=True)

    # ---- Time since last transaction (vectorized, fast) ----
    print("  Computing time since last transaction...")
    df["sender_time_since_last_txn"] = (
        df.groupby("nameOrig", sort=False)["_ts"].diff().dt.total_seconds().div(3600).fillna(-1.0)
    )

    # ---- Historical mean per sender (vectorized via cumsum/cumcount) ----
    # mean_at_row_i = sum_of_previous_rows / count_of_previous_rows
    # This is equivalent to expanding().mean().shift(1) but ~50x faster.
    print("  Computing historical mean per sender...")
    grp = df.groupby("nameOrig", sort=False)
    cum_sum_inclusive = grp["amount"].cumsum()
    cum_count_inclusive = grp.cumcount() + 1  # 1-indexed count of rows so far

    # Exclude current row: subtract current value, decrement count
    cum_sum_prior = cum_sum_inclusive - df["amount"]
    cum_count_prior = cum_count_inclusive - 1

    df["sender_amount_mean_historical"] = np.where(
        cum_count_prior > 0,
        cum_sum_prior / cum_count_prior.replace(0, 1),  # avoid div-by-zero
        0.0,
    )

    # ---- Rolling 1h and 24h windows ----
    # The genuinely tricky ones. We use the indexed groupby-rolling pattern,
    # which is much faster than apply because pandas vectorizes per group.
    print("  Computing 1h transaction counts...")
    df["sender_txn_count_1h"] = (
        df.set_index("_ts")
        .groupby("nameOrig", sort=False)["amount"]
        .rolling("1h", closed="left")
        .count()
        .reset_index(level=0, drop=True)
        .sort_index(kind="stable")
        .values
    )

    print("  Computing 24h transaction counts...")
    df["sender_txn_count_24h"] = (
        df.set_index("_ts")
        .groupby("nameOrig", sort=False)["amount"]
        .rolling("24h", closed="left")
        .count()
        .reset_index(level=0, drop=True)
        .sort_index(kind="stable")
        .values
    )

    print("  Computing 24h amount sums...")
    df["sender_amount_sum_24h"] = (
        df.set_index("_ts")
        .groupby("nameOrig", sort=False)["amount"]
        .rolling("24h", closed="left")
        .sum()
        .reset_index(level=0, drop=True)
        .sort_index(kind="stable")
        .values
    )

    # Fill the NaNs from rolling on the first row of each sender
    df["sender_txn_count_1h"] = df["sender_txn_count_1h"].fillna(0).astype("int32")
    df["sender_txn_count_24h"] = df["sender_txn_count_24h"].fillna(0).astype("int32")
    df["sender_amount_sum_24h"] = df["sender_amount_sum_24h"].fillna(0.0)

    # ---- Ratios (per-row, trivially vectorized) ----
    print("  Computing balance ratios...")
    df["amount_to_oldbalance_ratio"] = (
        df["amount"] / df["oldbalanceOrg"].replace(0, np.nan)
    ).fillna(-1.0)

    df["drains_origin"] = ((df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)).astype("int8")

    # Restore chronological order across the whole dataset
    df = df.sort_values(TIME_COLUMN).reset_index(drop=True)

    # Drop helper columns
    df = df.drop(columns=["_ts", "nameOrig", "nameDest"])

    return df


# Cache the result
cache_path = DATA_INTERIM_DIR / "paysim_with_features.parquet"

if cache_path.exists():
    print(f"Loading cached features from {cache_path}")
    df_feat = pd.read_parquet(cache_path)
else:
    print("Engineering features (this takes 2-5 minutes)...")
    df_feat = engineer_features(df)
    print(f"\nSaving cached features to {cache_path}")
    df_feat.to_parquet(cache_path, index=False)

print(f"\nFinal shape: {df_feat.shape}")
print(f"Columns: {df_feat.columns.tolist()}")

## Sanity check: did we leak?

A simple test: pick a fraud transaction in the data, find the *first* time its sender appears, and check the rolling features. They should be 0 (no prior activity), -1 (no prior balance), or sentinel values — never large numbers that would imply the model saw the future.

In [ ]:
# Find a sender's first-ever transaction and check its features.
# All rolling features for a "first ever" row should be 0 or sentinel.
first_appearances = df_feat.loc[df_feat.groupby(df["nameOrig"]).head(1).index]
first_fraud_appearance = first_appearances[first_appearances[LABEL_COLUMN] == 1].head(1)

print("First-ever transaction for a fraud sender (rolling features should be ~zero):")
print(
    first_fraud_appearance[
        [
            "step",
            "sender_txn_count_1h",
            "sender_txn_count_24h",
            "sender_amount_sum_24h",
            "sender_amount_mean_historical",
            "sender_time_since_last_txn",
            "isFraud",
        ]
    ].T
)

## Time-based split (same as notebook 02)

In [ ]:
train_df, test_df = time_based_split(df_feat, test_fraction=0.2)
X_train, y_train = split_features_label(train_df)
X_test, y_test = split_features_label(test_df)

print(f"Train: {len(X_train):,}  ({y_train.mean():.4%} fraud)")
print(f"Test:  {len(X_test):,}  ({y_test.mean():.4%} fraud)")
print(f"\nFeature columns: {X_train.columns.tolist()}")

In [ ]:
train_df, test_df = time_based_split(df_feat, test_fraction=0.2)
X_train, y_train = split_features_label(train_df)
X_test, y_test = split_features_label(test_df)

print(f"Train: {len(X_train):,}  ({y_train.mean():.4%} fraud)")
print(f"Test:  {len(X_test):,}  ({y_test.mean():.4%} fraud)")
print(f"\nFeature columns: {X_train.columns.tolist()}")

## Train XGBoost on enriched features, log to MLflow

Same XGBoost hyperparameters as notebook 02, so the *only* variable that changed is the feature set. Any improvement is attributable to features, not to model tuning. This is good experimental hygiene.

In [ ]:
categorical_cols = ["type"]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

with mlflow.start_run(run_name="xgboost_enriched") as run:
    mlflow.log_params(
        {
            "model_type": "xgboost",
            "feature_set": "enriched_v1",
            "n_engineered_features": 7,
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.1,
            "scale_pos_weight": scale_pos_weight,
            "tree_method": "hist",
            "random_state": RANDOM_SEED,
        }
    )

    model = Pipeline(
        [
            ("prep", preprocessor),
            (
                "clf",
                xgb.XGBClassifier(
                    n_estimators=200,
                    max_depth=6,
                    learning_rate=0.1,
                    scale_pos_weight=scale_pos_weight,
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                    tree_method="hist",
                    eval_metric="aucpr",
                ),
            ),
        ]
    )
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    metrics_default = evaluate_at_threshold(y_test.to_numpy(), proba, threshold=0.5)
    best_threshold = find_threshold_for_precision(y_test.to_numpy(), proba, min_precision=0.95)
    metrics_calibrated = evaluate_at_threshold(y_test.to_numpy(), proba, threshold=best_threshold)

    mlflow.log_metrics(
        {
            "precision_at_0.5": metrics_default.precision,
            "recall_at_0.5": metrics_default.recall,
            "f1_at_0.5": metrics_default.f1,
            "pr_auc": metrics_default.pr_auc,
            "roc_auc": metrics_default.roc_auc,
            "calibrated_threshold": best_threshold,
            "precision_calibrated": metrics_calibrated.precision,
            "recall_calibrated": metrics_calibrated.recall,
            "f1_calibrated": metrics_calibrated.f1,
        }
    )

    mlflow.sklearn.log_model(model, artifact_path="model")
    new_run_id = run.info.run_id

print(f"\nEnriched XGBoost run logged: {new_run_id}")
print(f"\nMetrics at calibrated threshold {best_threshold:.4f}:")
print(metrics_calibrated.pretty_print())

## Compare against the current production model

We fetch the production model's metrics from MLflow's registry, compare on both gates, and decide whether to promote.

**Promotion gate:**
- `pr_auc_new ≥ pr_auc_prod` (curve must dominate)
- `f1_calibrated_new ≥ f1_calibrated_prod` (operating point must improve)

Both must hold. Equal counts as "win" so that retraining on fresh data still promotes when the model is otherwise identical — a real production system needs to refresh on new data even when metrics don't change.

In [ ]:
client = MlflowClient()

# Resolve the current production version
prod_version = client.get_model_version_by_alias(
    name="fraud-detector",
    alias="production",
)
prod_run = client.get_run(prod_version.run_id)

prod_pr_auc = prod_run.data.metrics["pr_auc"]
prod_f1_calibrated = prod_run.data.metrics["f1_calibrated"]

print(f"Current production model: fraud-detector v{prod_version.version}")
print(f"  Source run: {prod_version.run_id}")
print(f"  pr_auc:         {prod_pr_auc:.4f}")
print(f"  f1_calibrated:  {prod_f1_calibrated:.4f}")
print()
print(f"New (enriched) run: {new_run_id}")
print(f"  pr_auc:         {metrics_default.pr_auc:.4f}")
print(f"  f1_calibrated:  {metrics_calibrated.f1:.4f}")

## Conditional promotion

This cell is the heart of the MLOps loop. If the new model beats production on both gates, it gets registered as a new version and the `production` alias moves to it. If not, the run is logged but production is untouched.

In [ ]:
beats_pr_auc = metrics_default.pr_auc >= prod_pr_auc
beats_f1 = metrics_calibrated.f1 >= prod_f1_calibrated

print("Gates:")
print(f"  pr_auc:        {metrics_default.pr_auc:.4f} >= {prod_pr_auc:.4f}  →  {beats_pr_auc}")
print(f"  f1_calibrated: {metrics_calibrated.f1:.4f} >= {prod_f1_calibrated:.4f}  →  {beats_f1}")

if beats_pr_auc and beats_f1:
    print("\n✓ New model passes all gates. Promoting to production.")

    # Register the new version
    new_version = mlflow.register_model(
        model_uri=f"runs:/{new_run_id}/model",
        name="fraud-detector",
    )
    # Add the threshold and metric tags (same pattern as notebook 02)
    client.set_model_version_tag(
        name="fraud-detector",
        version=new_version.version,
        key="calibrated_threshold",
        value=str(best_threshold),
    )
    client.set_model_version_tag(
        name="fraud-detector",
        version=new_version.version,
        key="precision_at_calibration",
        value=str(metrics_calibrated.precision),
    )
    client.set_model_version_tag(
        name="fraud-detector",
        version=new_version.version,
        key="recall_at_calibration",
        value=str(metrics_calibrated.recall),
    )
    client.set_model_version_tag(
        name="fraud-detector",
        version=new_version.version,
        key="feature_set",
        value="enriched_v1",
    )

    # Move the production alias to the new version
    client.set_registered_model_alias(
        name="fraud-detector",
        alias="production",
        version=new_version.version,
    )

    print(f"✓ Registered fraud-detector v{new_version.version}")
    print(f"✓ 'production' alias now points to v{new_version.version}")
    print(f"  Previous production (v{prod_version.version}) remains available for rollback.")
else:
    print(
        "\n✗ New model does not beat current production. Run logged for posterity, alias unchanged."
    )
    print(f"  Production remains: fraud-detector v{prod_version.version}")

## Verify the new production model loads correctly

Same load-by-alias pattern as notebook 02. If features changed but production code stays the same — this is exactly what a production system needs.

In [ ]:
loaded_model = mlflow.sklearn.load_model("models:/fraud-detector@production")
prod_version_now = client.get_model_version_by_alias(
    name="fraud-detector",
    alias="production",
)
production_threshold_now = float(prod_version_now.tags["calibrated_threshold"])

print("✓ Loaded model from alias 'production'")
print(f"  Resolved to version {prod_version_now.version}")
print(f"  Feature set: {prod_version_now.tags.get('feature_set', 'unknown')}")
print(f"  Threshold: {production_threshold_now:.4f}")

# Sanity check on test samples
sample_proba = loaded_model.predict_proba(X_test.head(5))[:, 1]
sample_predictions = (sample_proba >= production_threshold_now).astype(int)
print(f"\n  Sample predictions: {sample_predictions.tolist()}")
print(f"  Sample probabilities: {sample_proba.round(4).tolist()}")

## What you have now

1. Seven engineered features built with lookahead-leakage protection.
2. A new model trained on those features, logged to MLflow.
3. A programmatic promotion gate that only updates production when the new model wins on both PR-AUC and F1.
4. The production alias now points at the enriched model (assuming it won — check the printed output).
5. Version 1 and version 2 are both retained in the registry, available for rollback by changing the alias.

This is the complete MLOps training loop: train → log → compare → conditionally promote. Every later component (Lambda, monitoring, retraining flows) consumes this same model registry. The contract is `models:/fraud-detector@production`.

**Next:** Week 3 — Dockerize the inference path. Build a FastAPI service that loads the production model on startup, exposes a `/predict` endpoint, and serves predictions. This is the foundation for the Lambda deployment in week 4.